## Con diccionario

In [1]:
# Ruta del archivo de diccionario
ruta_archivo_datos = r"c:\Users\c50012218\Documents\diccionario.xlsx"

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

def formatear_precio(precio_raw):
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    # Extrae el código de producto: el último segmento antes del ? en la URL
    match = re.search(r'/([^/?]+)\?', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    # Extrae el SKU del parámetro skuid en la URL
    match = re.search(r'skuid=([^&]+)', url)
    return match.group(1) if match else ""

def extraer_detalle_producto(url, options):
    resultados = []
    try:
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h3.attributeAccordionText"))
        )
        soup = BeautifulSoup(driver.page_source, "html.parser")
        color_list = soup.find("ul", id="color-list-mobile")
        color_buttons = color_list.find_all("a", class_="atom-color") if color_list else [None]
        size_list = soup.find("ul", id="size-list-container")
        size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]

        combinaciones_unicas = set()
        if color_buttons == [None] and size_buttons == [None]:
            color = ""
            codigo = ""
            tamano = ""
            precio_normal = ""
            precio_desc = ""
            color_tag = soup.find("h3", class_="attributeAccordionText")
            if color_tag and "Color:" in color_tag.text:
                color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                color = color_span.text.strip() if color_span else ""
            codigo_tag = soup.find("p", class_="product-header-subcontainer-information--code")
            if codigo_tag:
                codigo_span = codigo_tag.find("span")
                codigo = codigo_span.text.strip() if codigo_span else ""
            tamano_tag = None
            for h3 in soup.find_all("h3", class_="attributeAccordionText"):
                if "Tamaño:" in h3.text:
                    tamano_tag = h3
                    break
            if tamano_tag:
                tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                tamano = tamano_span.text.strip() if tamano_span else ""
            precio_normal_tag = soup.find("p", class_="a-product__paragraphRegularPrice")
            precio_desc_tag = soup.find("p", class_="a-product__paragraphDiscountPrice")
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            clave = (color, tamano, codigo)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((color, codigo, tamano, precio_normal, precio_desc))
        else:
            for color_btn in color_buttons:
                if color_btn:
                    color_skuid = color_btn.get("data-skuid")
                    try:
                        driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//a[@data-skuid='{color_skuid}']"))
                        time.sleep(2)
                    except Exception:
                        pass
                soup_color = BeautifulSoup(driver.page_source, "html.parser")
                size_list = soup_color.find("ul", id="size-list-container")
                size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]
                for size_btn in size_buttons:
                    if size_btn and '-disabled' not in size_btn.get('class', []):
                        size_skuid = size_btn.get("data-skuid")
                        try:
                            driver.execute_script("arguments[0].click();", driver.find_element(By.XPATH, f"//button[@data-skuid='{size_skuid}']"))
                            time.sleep(2)
                        except Exception:
                            pass
                    soup_var = BeautifulSoup(driver.page_source, "html.parser")
                    if soup_var.find("p", class_="a-productInfo_selledByLink"):
                        continue
                    color = ""
                    codigo = ""
                    tamano = ""
                    precio_normal = ""
                    precio_desc = ""
                    color_tag = soup_var.find("h3", class_="attributeAccordionText")
                    if color_tag and "Color:" in color_tag.text:
                        color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                        color = color_span.text.strip() if color_span else ""
                    codigo_tag = soup_var.find("p", class_="product-header-subcontainer-information--code")
                    if codigo_tag:
                        codigo_span = codigo_tag.find("span")
                        codigo = codigo_span.text.strip() if codigo_span else ""
                    tamano_tag = None
                    for h3 in soup_var.find_all("h3", class_="attributeAccordionText"):
                        if "Tamaño:" in h3.text:
                            tamano_tag = h3
                            break
                    if tamano_tag:
                        tamano_span = tamano_tag.find("span", class_="attributeAccordionTextValue")
                        tamano = tamano_span.text.strip() if tamano_span else ""
                    precio_normal_tag = soup_var.find("p", class_="a-product__paragraphRegularPrice")
                    precio_desc_tag = soup_var.find("p", class_="a-product__paragraphDiscountPrice")
                    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
                    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
                    clave = (color, tamano, codigo)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((color, codigo, tamano, precio_normal, precio_desc))
        driver.quit()
    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    return resultados

def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch"]
    return any(p in nombre for p in palabras)

options = Options()
options.add_argument("--disable-gpu")
#options.add_argument("--HEADLESS")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []
visited_urls = set()

driver = webdriver.Chrome(options=options)
driver.get("https://www.suburbia.com.mx/tienda/celulares/N-7%2FJAomi6%2F9idzzftUHxlt8R%2BU%2FBc9OY0haLrErIDpijPCjGLLo4h8IQRLeNZbVha")
time.sleep(10)
soup = BeautifulSoup(driver.page_source, "html.parser")
products = soup.find_all("li", class_="m-product__card")
print(f"Productos encontrados: {len(products)}")
for product in products:
    marca = product.find("h3", class_="a-card-brand")
    nombre = product.find("h3", class_="card-title a-card-description")
    nombre_texto = nombre.text.strip() if nombre else ""
    if es_smartwatch(nombre_texto):
        continue
    precio_normal = product.find("p", class_="a-card-price")
    precio_desc = product.find("p", class_="a-card-discount")
    a_tag = product.find("a", href=True)
    url_cel = ""
    if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
        url_cel = "https://www.suburbia.com.mx" + a_tag["href"]
    color_div = product.find("div", class_="a-plp-color")
    colores = []
    if color_div:
        for span in color_div.find_all("span", class_="atom-color"):
            color_nombre = span.get("title", "")
            color_hex = span.get("data-color", "")
            colores.append((color_nombre, color_hex))
    else:
        colores = [("", "")]
    precio_normal_text = precio_normal.text.strip() if precio_normal else ""
    precio_desc_text = precio_desc.text.strip() if precio_desc else ""
    # Si solo hay precio en descuento, guárdalo en precio_normal y "No discount" en descuento
    if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "",  # Se llenará después desde el diccionario
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": formatear_precio(precio_desc_text),
                "Precio_descuento": "No discount"
            })
    elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_urls:
        variantes = extraer_detalle_producto(url_cel, options)
        visited_urls.add(url_cel)
        combinaciones_unicas = set()
        for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
            clave = (color_det, tamano_det, extraer_sku(url_cel))
            if clave in combinaciones_unicas:
                continue
            combinaciones_unicas.add(clave)
            # Si solo hay precio en descuento, guárdalo en precio_normal y "No discount" en descuento
            if not precio_normal_det and precio_desc_det:
                precio_normal_final = formatear_precio(precio_desc_det)
                precio_desc_final = "No discount"
            else:
                precio_normal_final = formatear_precio(precio_normal_det)
                precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_det,
                "Capacidad": tamano_det,  # Capacidad extraída de la web, puede ser sobrescrita por el diccionario
                "CodigoProducto": codigo_det,
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": precio_normal_final,
                "Precio_descuento": precio_desc_final
            })
    else:
        for color_nombre, color_hex in colores:
            final_data.append({
                "Marca": marca.text.strip() if marca else "",
                "Nombre": nombre.text.strip() if nombre else "",
                "Color": color_nombre,
                "Capacidad": "",  # Se llenará después desde el diccionario
                "CodigoProducto": extraer_codigo_producto(url_cel),
                "SKU": extraer_sku(url_cel),
                "URL": url_cel,
                "FechaConsulta": fecha_consulta,
                "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount"
            })
driver.quit()

def scrape_page(page):
    driver = webdriver.Chrome(options=options)
    url = f"https://www.suburbia.com.mx/tienda/celulares/N-7%2FJAomi6%2F9idzzftUHxlt8R%2BU%2FBc9OY0haLrErIDpijPCjGLLo4h8IQRLeNZbVha/page-{page}"
    driver.get(url)
    time.sleep(9)
    for _ in range(2):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.5)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    products = soup.find_all("li", class_="m-product__card")
    print(f"Productos encontrados en página {page}: {len(products)}")
    page_data = []
    for product in products:
        marca = product.find("h3", class_="a-card-brand")
        nombre = product.find("h3", class_="card-title a-card-description")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        precio_normal = product.find("p", class_="a-card-price")
        precio_desc = product.find("p", class_="a-card-discount")
        a_tag = product.find("a", href=True)
        url_cel = ""
        if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
            url_cel = "https://www.suburbia.com.mx" + a_tag["href"]
        color_div = product.find("div", class_="a-plp-color")
        colores = []
        if color_div:
            for span in color_div.find_all("span", class_="atom-color"):
                color_nombre = span.get("title", "")
                color_hex = span.get("data-color", "")
                colores.append((color_nombre, color_hex))
        else:
            colores = [("", "")]
        precio_normal_text = precio_normal.text.strip() if precio_normal else ""
        precio_desc_text = precio_desc.text.strip() if precio_desc else ""
        if (not precio_normal or not precio_normal_text) and precio_desc and precio_desc_text:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",  # Se llenará después desde el diccionario
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_desc_text),
                    "Precio_descuento": "No discount"
                })
        elif ('-' in precio_normal_text or '-' in precio_desc_text) and url_cel not in visited_urls:
            variantes = extraer_detalle_producto(url_cel, options)
            visited_urls.add(url_cel)
            combinaciones_unicas = set()
            for color_det, codigo_det, tamano_det, precio_normal_det, precio_desc_det in variantes:
                clave = (color_det, tamano_det, extraer_sku(url_cel))
                if clave in combinaciones_unicas:
                    continue
                combinaciones_unicas.add(clave)
                if not precio_normal_det and precio_desc_det:
                    precio_normal_final = formatear_precio(precio_desc_det)
                    precio_desc_final = "No discount"
                else:
                    precio_normal_final = formatear_precio(precio_normal_det)
                    precio_desc_final = formatear_precio(precio_desc_det) if precio_desc_det else "No discount"
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_det,
                    "Capacidad": tamano_det,  # Capacidad extraída de la web, puede ser sobrescrita por el diccionario
                    "CodigoProducto": codigo_det,
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": precio_normal_final,
                    "Precio_descuento": precio_desc_final
                })
        else:
            for color_nombre, color_hex in colores:
                page_data.append({
                    "Marca": marca.text.strip() if marca else "",
                    "Nombre": nombre.text.strip() if nombre else "",
                    "Color": color_nombre,
                    "Capacidad": "",  # Se llenará después desde el diccionario
                    "CodigoProducto": extraer_codigo_producto(url_cel),
                    "SKU": extraer_sku(url_cel),
                    "URL": url_cel,
                    "FechaConsulta": fecha_consulta,
                    "Precio_normal": formatear_precio(precio_normal_text) if precio_normal else "",
                    "Precio_descuento": formatear_precio(precio_desc_text) if precio_desc else "No discount"
                })
    driver.quit()
    return page_data

pages = list(range(2, 5))
with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(scrape_page, pages)
    for page_data in results:
        final_data.extend(page_data)

if final_data:
    df = pd.DataFrame(final_data)
    print(f"\n=== EXTRACCIÓN COMPLETADA ===")
    print(f"Total de productos extraídos: {len(df)}")
    print(f"SKUs únicos extraídos: {df['SKU'].nunique()}")
    
    # Cargar diccionario desde Excel y asignar capacidades
    try:
        diccionario = pd.read_excel(ruta_archivo_datos, sheet_name="Suburbia")
        print(f"\n=== CARGA DE DICCIONARIO ===")
        print(f"Diccionario cargado exitosamente con {len(diccionario)} registros")
        
        # Normalizar nombres de columnas
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "capacidad" in diccionario.columns and "Capacidad" not in diccionario.columns:
            diccionario.rename(columns={"capacidad": "Capacidad"}, inplace=True)
            
        print(f"Columnas disponibles en el diccionario: {list(diccionario.columns)}")
        
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            print(f"\n=== PROCESAMIENTO DE SKUs ===")
            # Mostrar algunos ejemplos de SKUs antes del procesamiento
            print(f"Ejemplos de SKUs en datos extraídos: {df['SKU'].head(5).tolist()}")
            print(f"Ejemplos de SKUs en diccionario: {diccionario['SKU'].head(5).tolist()}")
            
            # Limpiar y preparar datos para el merge - ser más agresivo con la limpieza
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip().str.lower()
            df["SKU"] = df["SKU"].astype(str).str.strip().str.lower()
            
            # Verificar que no haya SKUs vacíos
            df = df[df["SKU"] != ""]
            diccionario = diccionario[diccionario["SKU"] != ""]
            
            # Crear diccionario limpio solo con SKU y Capacidad, filtrar valores nulos
            diccionario_limpio = diccionario[["SKU", "Capacidad"]].dropna().drop_duplicates()
            print(f"Registros únicos válidos en diccionario: {len(diccionario_limpio)}")
            
            # Mostrar coincidencias antes del merge
            skus_en_datos = set(df["SKU"].unique())
            skus_en_diccionario = set(diccionario_limpio["SKU"].unique())
            coincidencias = skus_en_datos.intersection(skus_en_diccionario)
            print(f"SKUs que coinciden entre datos y diccionario: {len(coincidencias)}")
            print(f"Algunos SKUs coincidentes: {list(coincidencias)[:5]}")
            
            print(f"\n=== ASIGNACIÓN DE CAPACIDADES ===")
            # Realizar merge para asignar capacidades
            df_antes = len(df)
            df = df.merge(diccionario_limpio, on="SKU", how="left", suffixes=('_orig', '_dict'))
            
            # Actualizar capacidades: priorizar del diccionario, luego la extraída
            df["Capacidad"] = df["Capacidad_dict"].fillna(df["Capacidad_orig"])
            df.drop(columns=["Capacidad_orig", "Capacidad_dict"], inplace=True)
            
            # Estadísticas detalladas del merge
            capacidades_asignadas = df["Capacidad"].notna().sum()
            capacidades_desde_excel = (df["SKU"].isin(diccionario_limpio["SKU"]) & df["Capacidad"].notna()).sum()
            capacidades_vacias = df["Capacidad"].isna().sum()
            
            print(f"Productos con capacidad asignada: {capacidades_asignadas}/{len(df)}")
            print(f"Capacidades obtenidas del Excel: {capacidades_desde_excel}")
            print(f"Productos sin capacidad: {capacidades_vacias}")
            
            # Mostrar algunos ejemplos de productos sin capacidad
            if capacidades_vacias > 0:
                productos_sin_capacidad = df[df["Capacidad"].isna()]["SKU"].head(5).tolist()
                print(f"Ejemplos de SKUs sin capacidad: {productos_sin_capacidad}")
            
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'.")
            print(f"Columnas encontradas: {list(diccionario.columns)}")
            
    except FileNotFoundError:
        print(f"Archivo '{ruta_archivo_datos}' no encontrado. Continuando sin diccionario de capacidades.")
    except Exception as e:
        print(f"Error cargando el diccionario: {e}. Continuando sin diccionario de capacidades.")
        import traceback
        traceback.print_exc()
    
    print(f"\n=== PROCESAMIENTO FINAL ===")
    # Calcula el porcentaje de descuento
    df["Porcentaje_descuento"] = df.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )
    
    columnas_final = [
            "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
            "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento"
        ]
    df = df[columnas_final]

    # Elimina duplicados
    df = df.drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"suburbia_{fecha_hoy}.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"Datos exportados a {nombre_archivo}")
    print(f"Total de productos encontrados: {len(df)}")
else:
    print("No se encontraron productos. Ajusta el selector de productos si es necesario.")

Productos encontrados: 56
Productos encontrados en página 3: 56
Productos encontrados en página 4: 29
Productos encontrados en página 2: 56

=== EXTRACCIÓN COMPLETADA ===
Total de productos extraídos: 349
SKUs únicos extraídos: 194

=== CARGA DE DICCIONARIO ===
Diccionario cargado exitosamente con 686 registros
Columnas disponibles en el diccionario: ['Carrier', 'Marca', 'Nombre', 'Color', 'Capacidad', 'Red', 'CodigoProducto', 'SKU', 'URL', 'Nombre homologado', 'Nombre final', 'Modelo', 'Unnamed: 12', 'Unnamed: 13']

=== PROCESAMIENTO DE SKUs ===
Ejemplos de SKUs en datos extraídos: ['sb5013416801', 'sb5013416801', 'sb5014131002', 'sb5014131002', 'sb5014131002']
Ejemplos de SKUs en diccionario: ['sb5009575679', 'sb5009575679', 'sb5009719794', 'sb5009719794', 'sb5011252046']
Registros únicos válidos en diccionario: 405
SKUs que coinciden entre datos y diccionario: 189
Algunos SKUs coincidentes: ['sb5011547054', 'sb5012674785', 'sb5013207978', 'sb5011671499', 'sb5013758168']

=== ASIGNAC

## Completo

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

def formatear_precio(precio_raw):
    if not precio_raw:
        return ""
    precios = [re.sub(r'[^\d-]', '', p) for p in precio_raw.split('-')]
    precios = [p for p in precios if p]
    if len(precios) == 2:
        return f"{formatear_simple(precios[0])} - {formatear_simple(precios[1])}"
    elif len(precios) == 1:
        return formatear_simple(precios[0])
    else:
        return ""

def formatear_simple(precio):
    if len(precio) > 2:
        return f"${int(precio[:-2]):,}.{precio[-2:]}"
    elif len(precio) == 2:
        return f"$0.{precio}"
    elif len(precio) == 1:
        return f"$0.0{precio}"
    else:
        return ""

def calcular_porcentaje_descuento(precio_normal, precio_descuento):
    try:
        p_ori = float(re.sub(r'[^\d.]', '', precio_normal).replace(',', ''))
        p_desc = float(re.sub(r'[^\d.]', '', precio_descuento).replace(',', ''))
        if p_ori > 0 and p_desc > 0 and p_ori > p_desc:
            return f"{round(100 * (p_ori - p_desc) / p_ori, 2)}%"
        elif p_desc > 0:
            return "0%"
    except Exception:
        pass
    return "No discount"

def extraer_codigo_producto(url):
    # Extrae el código de producto: el último segmento antes del ? en la URL
    match = re.search(r'/([^/?]+)\?', url)
    return match.group(1) if match else ""

def extraer_sku(url):
    # Extrae el SKU del parámetro skuid en la URL
    match = re.search(r'skuid=([^&]+)', url)
    return match.group(1) if match else ""

def es_smartwatch(nombre):
    nombre = (nombre or "").lower()
    palabras = ["smartwatch", "watch", "reloj"]
    return any(p in nombre for p in palabras)

def crear_driver_options():
    """Crea las opciones del navegador"""
    options = Options()
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    #options.add_argument("--headless")  # Modo headless para mejor rendimiento en paralelo
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")
    return options

def extraer_detalle_producto_completo(url):
    """Extrae toda la información detallada de un producto individual"""
    resultados = []
    driver = None
    try:
        options = crear_driver_options()
        driver = webdriver.Chrome(options=options)
        driver.get(url)
        
        # Esperar a que cargue la página
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
        )
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # Extraer información básica del producto
        marca = ""
        nombre = ""
        
        # Buscar marca y nombre
        marca_tag = soup.find("p", class_="brandTitleOffBrand")
        if not marca_tag:
            marca_tag = soup.find("span", class_="a-product__brand")
        if marca_tag:
            marca = marca_tag.text.strip()
        
        nombre_tag = soup.find("h1", class_="a-product__information--title")
        if nombre_tag:
            nombre = nombre_tag.text.strip()
        
        # Si no encuentra con esas clases, buscar alternativas
        if not nombre:
            nombre_tag = soup.find("h1")
            if nombre_tag:
                nombre = nombre_tag.text.strip()
        
        # Buscar listas de colores y tamaños
        color_list = soup.find("ul", id="color-list-mobile")
        color_buttons = color_list.find_all("a", class_="atom-color") if color_list else [None]
        
        size_list = soup.find("ul", id="size-list-container")
        size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]

        combinaciones_unicas = set()
        
        # Si no hay variantes de color o tamaño
        if color_buttons == [None] and size_buttons == [None]:
            color = ""
            codigo = ""
            capacidad = ""
            precio_normal = ""
            precio_desc = ""
            
            # Extraer color si está disponible
            color_tag = soup.find("h3", class_="attributeAccordionText")
            if color_tag and "Color:" in color_tag.text:
                color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                color = color_span.text.strip() if color_span else ""
            
            # Extraer código del producto
            codigo_tag = soup.find("p", class_="product-header-subcontainer-information--code")
            if codigo_tag:
                codigo_span = codigo_tag.find("span")
                codigo = codigo_span.text.strip() if codigo_span else ""
            
            # Extraer capacidad/tamaño
            capacidad_tag = None
            for h3 in soup.find_all("h3", class_="attributeAccordionText"):
                if any(keyword in h3.text for keyword in ["Tamaño:", "Capacidad:", "Almacenamiento:"]):
                    capacidad_tag = h3
                    break
            if capacidad_tag:
                capacidad_span = capacidad_tag.find("span", class_="attributeAccordionTextValue")
                capacidad = capacidad_span.text.strip() if capacidad_span else ""
            
            # Extraer precios
            precio_normal_tag = soup.find("p", class_="a-product__paragraphRegularPrice")
            precio_desc_tag = soup.find("p", class_="a-product__paragraphDiscountPrice")
            
            precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
            precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
            
            # Si no hay precio normal pero sí precio de descuento, usar el de descuento como normal
            if not precio_normal and precio_desc:
                precio_normal = precio_desc
                precio_desc = ""
            
            clave = (color, capacidad, codigo)
            if clave not in combinaciones_unicas:
                combinaciones_unicas.add(clave)
                resultados.append((marca, nombre, color, codigo, capacidad, precio_normal, precio_desc))
        
        else:
            # Si hay variantes, iterar sobre todas las combinaciones
            for color_btn in color_buttons:
                if color_btn:
                    color_skuid = color_btn.get("data-skuid")
                    try:
                        driver.execute_script("arguments[0].click();", 
                                            driver.find_element(By.XPATH, f"//a[@data-skuid='{color_skuid}']"))
                        time.sleep(2)
                    except Exception:
                        pass
                
                # Actualizar soup después del click
                soup_color = BeautifulSoup(driver.page_source, "html.parser")
                size_list = soup_color.find("ul", id="size-list-container")
                size_buttons = size_list.find_all("button", class_="a-btn--actionpdp") if size_list else [None]
                
                for size_btn in size_buttons:
                    if size_btn and '-disabled' not in size_btn.get('class', []):
                        size_skuid = size_btn.get("data-skuid")
                        try:
                            driver.execute_script("arguments[0].click();", 
                                                driver.find_element(By.XPATH, f"//button[@data-skuid='{size_skuid}']"))
                            time.sleep(2)
                        except Exception:
                            pass
                    
                    # Extraer información de esta variante específica
                    soup_var = BeautifulSoup(driver.page_source, "html.parser")
                    
                    # Saltar si está vendido por terceros
                    if soup_var.find("p", class_="a-productInfo_selledByLink"):
                        continue
                    
                    color = ""
                    codigo = ""
                    capacidad = ""
                    precio_normal = ""
                    precio_desc = ""
                    
                    # Extraer información de esta variante
                    color_tag = soup_var.find("h3", class_="attributeAccordionText")
                    if color_tag and "Color:" in color_tag.text:
                        color_span = color_tag.find("span", class_="attributeAccordionTextValue")
                        color = color_span.text.strip() if color_span else ""
                    
                    codigo_tag = soup_var.find("p", class_="product-header-subcontainer-information--code")
                    if codigo_tag:
                        codigo_span = codigo_tag.find("span")
                        codigo = codigo_span.text.strip() if codigo_span else ""
                    
                    capacidad_tag = None
                    for h3 in soup_var.find_all("h3", class_="attributeAccordionText"):
                        if any(keyword in h3.text for keyword in ["Tamaño:", "Capacidad:", "Almacenamiento:"]):
                            capacidad_tag = h3
                            break
                    if capacidad_tag:
                        capacidad_span = capacidad_tag.find("span", class_="attributeAccordionTextValue")
                        capacidad = capacidad_span.text.strip() if capacidad_span else ""
                    
                    precio_normal_tag = soup_var.find("p", class_="a-product__paragraphRegularPrice")
                    precio_desc_tag = soup_var.find("p", class_="a-product__paragraphDiscountPrice")
                    
                    precio_normal = precio_normal_tag.get_text(strip=True) if precio_normal_tag else ""
                    precio_desc = precio_desc_tag.get_text(strip=True) if precio_desc_tag else ""
                    
                    if not precio_normal and precio_desc:
                        precio_normal = precio_desc
                        precio_desc = ""
                    
                    clave = (color, capacidad, codigo)
                    if clave not in combinaciones_unicas:
                        combinaciones_unicas.add(clave)
                        resultados.append((marca, nombre, color, codigo, capacidad, precio_normal, precio_desc))
        
    except Exception as e:
        print(f"Error extrayendo detalle de {url}: {e}")
    finally:
        if driver:
            try:
                driver.quit()
            except:
                pass
    
    return resultados

def procesar_producto(args):
    """Procesa un producto individual y retorna los datos (función para uso en paralelo)"""
    producto_url, fecha_consulta, indice, total = args
    thread_id = threading.current_thread().ident
    print(f"[Thread {thread_id}] Procesando producto {indice}/{total}: {producto_url}")
    
    variantes = extraer_detalle_producto_completo(producto_url)
    
    datos_producto = []
    for marca, nombre, color, codigo, capacidad, precio_normal, precio_desc in variantes:
        # Formatear precios
        precio_normal_formateado = formatear_precio(precio_normal) if precio_normal else ""
        precio_desc_formateado = formatear_precio(precio_desc) if precio_desc else "No discount"
        
        # Si no hay precio de descuento, ponerlo como "No discount"
        if not precio_desc_formateado or precio_desc_formateado == "":
            precio_desc_formateado = "No discount"
        
        datos_producto.append({
            "Marca": marca,
            "Nombre": nombre,
            "Color": color,
            "Capacidad": capacidad,
            "CodigoProducto": codigo,
            "SKU": extraer_sku(producto_url),
            "URL": producto_url,
            "FechaConsulta": fecha_consulta,
            "Precio_normal": precio_normal_formateado,
            "Precio_descuento": precio_desc_formateado
        })
    
    print(f"[Thread {thread_id}] Completado producto {indice}/{total}: {len(datos_producto)} variantes encontradas")
    return datos_producto

fecha_consulta = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
final_data = []
urls_productos = []

# Función para extraer URLs de productos de una página
def extraer_urls_pagina(page_num):
    options = crear_driver_options()
    driver = webdriver.Chrome(options=options)
    if page_num == 1:
        url = "https://www.suburbia.com.mx/tienda/celulares/N-7%2FJAomi6%2F9idzzftUHxlt8R%2BU%2FBc9OY0haLrErIDpijPCjGLLo4h8IQRLeNZbVha"
    else:
        url = f"https://www.suburbia.com.mx/tienda/celulares/N-7%2FJAomi6%2F9idzzftUHxlt8R%2BU%2FBc9OY0haLrErIDpijPCjGLLo4h8IQRLeNZbVha/page-{page_num}"
    
    driver.get(url)
    time.sleep(10)
    
    # Hacer scroll para cargar todos los productos
    for _ in range(3):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    products = soup.find_all("li", class_="m-product__card")
    
    urls_pagina = []
    for product in products:
        # Verificar si no es smartwatch
        nombre = product.find("h3", class_="card-title a-card-description")
        nombre_texto = nombre.text.strip() if nombre else ""
        if es_smartwatch(nombre_texto):
            continue
        
        # Extraer URL del producto
        a_tag = product.find("a", href=True)
        if a_tag and a_tag["href"].startswith("/tienda/pdp/"):
            url_producto = "https://www.suburbia.com.mx" + a_tag["href"]
            urls_pagina.append(url_producto)
    
    driver.quit()
    print(f"Página {page_num}: {len(urls_pagina)} productos encontrados")
    return urls_pagina

# Extraer URLs de las primeras 4 páginas en paralelo
print("Extrayendo URLs de productos...")
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(extraer_urls_pagina, page) for page in range(1, 5)]
    for future in as_completed(futures):
        try:
            urls_pagina = future.result()
            urls_productos.extend(urls_pagina)
        except Exception as e:
            print(f"Error extrayendo URLs de página: {e}")

# Eliminar duplicados
urls_productos = list(set(urls_productos))
print(f"Total de productos únicos a procesar: {len(urls_productos)}")

# Preparar argumentos para procesamiento en paralelo
args_productos = [
    (url, fecha_consulta, i+1, len(urls_productos)) 
    for i, url in enumerate(urls_productos)
]

# Procesar productos en paralelo
print("Iniciando extracción detallada de productos en paralelo...")
print(f"Usando hasta 6 hilos para el procesamiento")

with ThreadPoolExecutor(max_workers=3) as executor:
    # Enviar todas las tareas
    futures = [executor.submit(procesar_producto, args) for args in args_productos]
    
    # Recopilar resultados conforme se completan
    for future in as_completed(futures):
        try:
            datos_producto = future.result()
            final_data.extend(datos_producto)
        except Exception as e:
            print(f"Error procesando producto: {e}")

# Procesar datos y exportar
if final_data:
    df = pd.DataFrame(final_data)
    
    # Cargar diccionario desde Excel y asignar capacidades
    try:
        diccionario = pd.read_excel(r"c:\Users\c50012218\Documents\diccionario.xlsx", sheet_name="Suburbia")
        print(f"Diccionario cargado exitosamente con {len(diccionario)} registros")
        
        # Normalizar nombres de columnas
        if "sku" in diccionario.columns and "SKU" not in diccionario.columns:
            diccionario.rename(columns={"sku": "SKU"}, inplace=True)
        if "capacidad" in diccionario.columns and "Capacidad" not in diccionario.columns:
            diccionario.rename(columns={"capacidad": "Capacidad"}, inplace=True)
        
        if "SKU" in diccionario.columns and "Capacidad" in diccionario.columns:
            # Mostrar algunos ejemplos de SKUs antes del procesamiento
            print(f"Ejemplos de SKUs en datos extraídos: {df['SKU'].head(5).tolist()}")
            print(f"Ejemplos de SKUs en diccionario: {diccionario['SKU'].head(5).tolist()}")
            
            # Limpiar y preparar datos para el merge - ser más agresivo con la limpieza
            diccionario["SKU"] = diccionario["SKU"].astype(str).str.strip().str.lower()
            df["SKU"] = df["SKU"].astype(str).str.strip().str.lower()
            
            # Verificar que no haya SKUs vacíos
            df = df[df["SKU"] != ""]
            diccionario = diccionario[diccionario["SKU"] != ""]
            
            # Crear diccionario limpio solo con SKU y Capacidad, filtrar valores nulos
            diccionario_limpio = diccionario[["SKU", "Capacidad"]].dropna().drop_duplicates()
            print(f"Registros únicos válidos en diccionario: {len(diccionario_limpio)}")
            
            # Mostrar coincidencias antes del merge
            skus_en_datos = set(df["SKU"].unique())
            skus_en_diccionario = set(diccionario_limpio["SKU"].unique())
            coincidencias = skus_en_datos.intersection(skus_en_diccionario)
            print(f"SKUs que coinciden entre datos y diccionario: {len(coincidencias)}")
            
            # Realizar merge para asignar capacidades
            df = df.merge(diccionario_limpio, on="SKU", how="left", suffixes=('_orig', '_dict'))
            
            # Actualizar capacidades: priorizar del diccionario, luego la extraída
            df["Capacidad"] = df["Capacidad_dict"].fillna(df["Capacidad_orig"])
            df.drop(columns=["Capacidad_orig", "Capacidad_dict"], inplace=True)
            
            # Estadísticas detalladas del merge
            capacidades_asignadas = df["Capacidad"].notna().sum()
            capacidades_desde_excel = (df["SKU"].isin(diccionario_limpio["SKU"]) & df["Capacidad"].notna()).sum()
            capacidades_vacias = df["Capacidad"].isna().sum()
            
            print(f"Productos con capacidad asignada: {capacidades_asignadas}/{len(df)}")
            print(f"Capacidades obtenidas del Excel: {capacidades_desde_excel}")
            print(f"Productos sin capacidad: {capacidades_vacias}")
            
        else:
            print("Advertencia: El diccionario no tiene columnas 'SKU' y/o 'Capacidad'. No se agregará capacidad por SKU.")
    except FileNotFoundError:
        print("Archivo de diccionario no encontrado. Continuando sin diccionario de capacidades.")
    
    # Calcula el porcentaje de descuento
    df["Porcentaje_descuento"] = df.apply(
        lambda row: calcular_porcentaje_descuento(row["Precio_normal"], row["Precio_descuento"]), axis=1
    )
    
    columnas_final = [
        "Marca", "Nombre", "Color", "Capacidad", "CodigoProducto", "SKU", "URL",
        "FechaConsulta", "Precio_normal", "Precio_descuento", "Porcentaje_descuento"
    ]
    df = df[columnas_final]

    # Elimina duplicados
    df = df.drop_duplicates()

    fecha_hoy = datetime.now().strftime("%Y-%m-%d")
    nombre_archivo = f"suburbia_{fecha_hoy}.xlsx"
    df.to_excel(nombre_archivo, index=False)
    print(f"\n=== PROCESO COMPLETADO ===")
    print(f"Datos exportados a: {nombre_archivo}")
    print(f"Total de productos procesados: {len(df)}")
    print(f"Productos únicos: {df['URL'].nunique()}")
    print(f"Marcas encontradas: {', '.join(df['Marca'].unique())}")
else:
    print("No se encontraron productos. Verifica la configuración del scraper.")

Extrayendo URLs de productos...
Página 4: 21 productos encontrados
Página 4: 21 productos encontrados
Página 3: 56 productos encontrados
Página 2: 56 productos encontrados
Página 3: 56 productos encontrados
Página 2: 56 productos encontrados
Página 1: 56 productos encontrados
Total de productos únicos a procesar: 170
Iniciando extracción detallada de productos en paralelo...
Usando hasta 6 hilos para el procesamiento
[Thread 39420] Procesando producto 1/170: https://www.suburbia.com.mx/tienda/pdp/honor-x7b-amoled-6.8-pulgadas/sb5009575661?skuid=sb5009575679
[Thread 57636] Procesando producto 2/170: https://www.suburbia.com.mx/tienda/pdp/motorola-moto-edge-50-neo-poled-6.3-pulgadas/sb5011252029?skuid=sb5011252046
[Thread 64288] Procesando producto 3/170: https://www.suburbia.com.mx/tienda/pdp/oppo-reno-11-amoled-6.7-pulgadas-movistar/sb5009719786?skuid=sb5009719794
Página 1: 56 productos encontrados
Total de productos únicos a procesar: 170
Iniciando extracción detallada de productos en